![Notebook 1](images/architecture-high-level.svg)

# 1. Your eval passes and measures nothing

> A Deep Agent hides its subagents' work. Write the obvious check and it passes whether
> or not a researcher misbehaved. **Three pitfalls, each demonstrated, then checks that
> provably fail.**

📖 Story and gotchas: [`README.md`](README.md)
💻 Code this notebook imports: [`helpers/`](helpers/) (`research_agent.py`, `agentcore_evals.py`, `research_checks.py`)
➡️ Next: [`02_score_traces_with_agentcore_evaluations.ipynb`](02_score_traces_with_agentcore_evaluations.ipynb)

---

In [ ]:
%pip install -q "langchain-aws[tools]>=1.7.6" "deepagents>=0.7.13" "bedrock-agentcore>=1.23.0"

In [ ]:
import os, sys
sys.path.insert(0, "helpers")   # the modules this notebook imports
os.environ.setdefault("AWS_REGION", "us-west-2")
print("region:", os.environ["AWS_REGION"])

## Ground truth that cannot expire

To evaluate different companies, regenerate the dataset and re-run. The answer key, the
prompts and the subagent list all follow from it, so no code changes:
`python helpers/edgar_dataset.py --tickers CRWD ZS NET`

Sources are SEC 10-K financial statements pinned by **EDGAR accession number**, which
identifies a filing that can never be edited or withdrawn. Real browser, real page,
immutable. Graded values are *derived* from several extracted figures, so an agent
cannot pass from memory or by skipping its tools.

In [ ]:
from research_agent import ANSWER_KEY, COMPANIES, GROWTH_RANK, RULE_OF_40_RANK

for c in COMPANIES:
    print(f"{c.name:10s} FY{c.fiscal_year} to {c.period_end}  {c.income_statement_url}")
print()
for name, v in ANSWER_KEY.items():
    print(f"{name:10s} {v}")

Two rankings that disagree. **Rule of 40 puts Datadog first** even though Snowflake
leads on both revenue and growth, so an agent that reasons "biggest wins" is wrong.
Note also that Datadog's fiscal year ends a month before the other two, which any
honest comparison has to state.

In [ ]:
print("by rule of 40   :", RULE_OF_40_RANK)
print("by revenue growth:", GROWTH_RANK)

## Build the agent

One browser toolkit per company, so one MicroVM per company. That isolation is what makes
parallel browsing possible.

In [ ]:
from research_agent import build_model, build_research_agent, cleanup

agent, toolkits = await build_research_agent(build_model())
print(f"coordinator + {len(COMPANIES)} researchers + 1 analyst")

## Pitfall 1: the agent's final state hides its subagents

Run it, then list every tool name visible in `result["messages"]`.

In [ ]:
from research_checks import CHART_PROMPT

result = await agent.ainvoke(
    {"messages": [{"role": "user", "content": CHART_PROMPT}]},
    # 60, not LangGraph's default 25: a four-subagent fan-out that retries trips it.
    config={"configurable": {"thread_id": "pitfall-1"}, "recursion_limit": 60},
)

In [ ]:
from langchain_core.messages import AIMessage

visible = {tc["name"] for m in result["messages"] if isinstance(m, AIMessage)
           for tc in (m.tool_calls or [])}
print("visible:", sorted(visible))
for tool in ("navigate_browser", "extract_text", "execute_code", "take_screenshot"):
    print(f"  {tool:<18} {tool in visible}")

Every subagent tool is **False**. So this check is worthless:

```python
assert "take_screenshot" not in visible   # passes no matter what happened
```

## Record the trajectory instead

`trajectory_from_langchain_events` captures every tool call at every nesting level.
Notebook 2 swaps it for a span reader that is not framework specific, producing the
**same object**.

In [ ]:
from agentcore_evals import trajectory_from_langchain_events

trajectory, answer = await trajectory_from_langchain_events(
    agent, query=CHART_PROMPT, thread_id="record-1")
print(trajectory.timeline())

## Pitfall 2: parallelism is overlap, not counting

Three `task()` calls in one turn does not mean they ran together. Overlapping intervals
does.

In [ ]:
print("task() calls          :", len(trajectory.calls("task")))
print("peak concurrent task():", trajectory.max_concurrent("task"))
print("nested tools          :", sorted(
    {c.name for c in trajectory.tool_calls if trajectory.is_nested(c)}))

## Checks in two tiers

**correctness** fails the eval. **efficiency** is recorded, never fatal, because serial
delegation is slower rather than wrong. See `research_checks.py` for the list.

In [ ]:
from agentcore_evals import check_all, report
from research_checks import standard_checks

passed, results = check_all(standard_checks(), trajectory, answer)
report("full_comparison", trajectory, results, passed)

In [ ]:
await cleanup(toolkits)
print("MicroVM sessions released")

## Pitfall 3: a check that cannot fail is decoration

Break the behavior on purpose and confirm the check fails. Uses **stand-in** tools, so
it costs seconds and says nothing about AgentCore or answer accuracy.

In [ ]:
from negative_control import forced_screenshot_agent, forced_serial_agent

agent_shot = forced_screenshot_agent(build_model())
traj_shot, _ = await trajectory_from_langchain_events(
    agent_shot, query="Research company alpha.", thread_id="neg-1")
shots = len(traj_shot.calls("take_screenshot"))
print(f"screenshots observed: {shots}  ->  'no screenshots' fails: {shots > 0}")

In [ ]:
agent_serial = forced_serial_agent(build_model())
traj_serial, _ = await trajectory_from_langchain_events(
    agent_serial, query="Research alpha, beta and gamma.", thread_id="neg-2")
print(f"task() calls: {len(traj_serial.calls('task'))}, "
      f"peak concurrency: {traj_serial.max_concurrent('task')}")
print(f"'3-way parallel' fails: {traj_serial.max_concurrent('task') < 3}")

Case 2 is the point: **three `task()` calls, peak concurrency 1.** A check that counted
calls per turn would have scored that as parallel.

## Optional: LangSmith for the local loop

Skipped unless `LANGSMITH_API_KEY` is set. LangSmith is strong here — datasets,
experiment comparison, `@pytest.mark.langsmith` in CI. It does not read AgentCore spans,
so notebook 2 stays on the AgentCore path.

In [ ]:
# Optional cell: needs `pip install agentevals` and LANGSMITH_API_KEY. Guarded on both,
# so it degrades to a message rather than failing the notebook.
if not os.environ.get("LANGSMITH_API_KEY"):
    print("LANGSMITH_API_KEY not set, skipping")
else:
    try:
        from agentevals.trajectory.match import create_trajectory_match_evaluator
    except ModuleNotFoundError:
        print("pip install agentevals to try this")
    else:
        matcher = create_trajectory_match_evaluator(trajectory_match_mode="superset")
        print("agentevals ready: strict / unordered / subset / superset tool-sequence match")
        print("it does NOT measure concurrency, so max_concurrent stays hand-rolled")

## What notebook 1 established

- Pinned sources give ground truth that does not expire
- A Deep Agent's final state hides subagent tool calls
- Real parallelism is overlapping intervals
- Every check has been shown to fail when its behavior breaks

➡️ Next: [`02_score_traces_with_agentcore_evaluations.ipynb`](02_score_traces_with_agentcore_evaluations.ipynb) scores real
traces with AWS evaluators in seconds, no deploy needed.